# Join HOROSCOPE and cancer metadata
- Load PCAWG metadata table with information about sample names, histologies, DCC, purity, ploidy, WGD, CN
- Add HOROSCOPE results: CLUSTER_H1 and _H2 from the germline sample and HOR_length from germline and tumor sample
- Same process for PPCG

In [ ]:
import pandas as pd

In [ ]:
chromosomes = [
    'chr1', 'chr3', 'chr4', 'chr5', 'chr6', 
    'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 
    'chr12', 'chr16', 'chr17', 'chr18', 'chr19',
    'chr20'
]

In [ ]:
df_cancer = pd.read_csv("/g/korbel/hain/centromere/samplesheets/pcawg.tumor.normal.CN.tsv", sep="\t")
#df_cancer = pd.read_csv("/g/korbel/hain/centromere/samplesheets/ppcg.tumor.normal.CN.tsv", sep="\t")

In [ ]:
df_horoscope = pd.read_csv("centromere_genotyping.tsv", sep="\t")

In [ ]:
reformatted_data = []

for _, row in df_cancer.iterrows():
    
    ### gather invariate sample parameters
    sample_tumor = row["pickedTumor"]
    sample_germline = row["aliquot_id"]
    histology = row["histology_abbreviation"]
    dcc = row["dcc_project_code"]
    purity = row["purity"]
    ploidy = row["ploidy"]
    wgd = row["wgd_status"]
    
    # gather chromosome-specific parameters
    for chrom in chromosomes:
        
        cn = row[f"{chrom}_CN"]
        
        ### get clusters and germline HOR length
        df_germ = df_horoscope[df_horoscope["SAMPLE"].eq(sample_germline) & df_horoscope["CHROM"].eq(chrom)]
        if df_germ.shape[0] == 0:
            raise Exception(f"MISSING DATA for sample {sample_germline} and chromosome {chrom}")
        elif df_germ.shape[0] > 1:
            raise Exception(f"MULTIPLE DATA for sample {sample_germline} and chromosome {chrom}")
        else:
            cluster_h1 = df_germ["CLUSTER_H1"].values[0]
            cluster_h2 = df_germ["CLUSTER_H2"].values[0]
            hor_length_germline = df_germ["HOR_LENGTH"].values[0]
        
        ### get tumor HOR length
        df_tum = df_horoscope[df_horoscope["SAMPLE"].eq(sample_tumor) & df_horoscope["CHROM"].eq(chrom)]
        if df_tum.shape[0] == 0:
            raise Exception(f"MISSING DATA for sample {sample_tumor} and chromosome {chrom}")
        elif df_tum.shape[0] > 1:
            raise Exception(f"MULTIPLE DATA for sample {sample_tumor} and chromosome {chrom}")
        else:
            hor_length_tumor = df_tum["HOR_LENGTH"].values[0]

        reformatted_data.append({
            "SAMPLE_TUMOR": sample_tumor,
            "SAMPLE_GERMLINE": sample_germline,
            "HISTOLOGY": histology,
            "DCC_PROJECT_CODE": dcc,
            "PURITY": purity,
            "PLOIDY": ploidy,
            "WGD": wgd,
            "CHROM": chrom,
            "CN": cn,
            "CLUSTER_H1": cluster_h1,
            "CLUSTER_H2": cluster_h2,
            "GERMLINE_TOTAL_HOR_LENGTH_COV_2": hor_length_germline,
            "TUMOR_TOTAL_HOR_LENGTH_COV_2": hor_length_tumor
        })

### save to file
pd.DataFrame(reformatted_data).to_csv("pcawg.horoscope.tsv", sep="\t", index=False)